# 03 - Turn forecasts into a capacity exercise
**Question:** How does adding a buffer trade lower shortages for more excess capacity?

Run notebooks 01 and 02 first. For each hour, hypothetical rental slots = `ceil(prediction * (1 + buffer))`.
Compare buffers of 5%, 15%, and 30%. Select the lowest assumed cost on VALIDATION, then evaluate that
fixed buffer on test. This avoids choosing the policy on the same data used to report its result.

A rental slot is the ability to support one trip in an hour. It is not a physical bicycle.
There are no station locations, trip durations, or starting inventory in this dataset, so this exercise
cannot calculate a real fleet size, bike rebalancing plan, observed stockouts, or financial savings.


In [ ]:
from pathlib import Path
import sys
# Support opening the notebook from the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'mobility').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'mobility').is_dir():
    raise RuntimeError('Open Jupyter from the project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from mobility.data import OUTPUT, invalidate_outputs
import pandas as pd
import numpy as np
import json
from IPython.display import display

from mobility.evaluation import capacity_scenarios
invalidate_outputs(3)
validation = pd.read_csv(OUTPUT / 'validation_predictions.csv')
test = pd.read_csv(OUTPUT / 'test_predictions.csv')
# Teaching assumptions in arbitrary cost units, not measured dollars.
EXCESS_COST = 0.5
SHORTAGE_COST = 3.0
validation_scenarios = capacity_scenarios(validation.cnt, validation.prediction,
                                         EXCESS_COST, SHORTAGE_COST)
selected_buffer = int(validation_scenarios.loc[validation_scenarios.hypothetical_cost.idxmin(), 'buffer_pct'])
display(validation_scenarios.round(2))
print('Lowest validation cost among the three choices:', selected_buffer, '%')


## 1. Report the fixed policy on test
Shortage slots = `max(actual - slots, 0)`; excess slots = `max(slots - actual, 0)`.
Cost = `0.5 * total excess slots + 3 * total shortage slots`.
Covered rentals percentage describes this hypothetical scenario. It is not measured service improvement.
The best choice may change if the assumptions change. No buffer is hardcoded as "best."


In [ ]:
test_scenarios = capacity_scenarios(test.cnt, test.prediction, EXCESS_COST, SHORTAGE_COST)
validation_scenarios.to_csv(OUTPUT / 'capacity_validation.csv', index=False)
test_scenarios.to_csv(OUTPUT / 'capacity_test.csv', index=False)
chosen = test_scenarios[test_scenarios.buffer_pct == selected_buffer].iloc[0].to_dict()
summary = {'selected_on': 'validation hypothetical cost', 'buffer_pct': selected_buffer,
           'assumptions': {'excess_cost_per_slot': EXCESS_COST, 'shortage_cost_per_slot': SHORTAGE_COST,
                           'cost_units': 'arbitrary teaching units'},
           'test_selected_policy': chosen}
(OUTPUT / 'capacity.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
display(test_scenarios.round(2))
print('Frozen validation-selected policy on test:')
display(pd.DataFrame([chosen]).round(2))


## 2. Check sensitivity using validation only
Change the penalty for a shortage from 1 to 6 cost units. This is a separate sensitivity exercise;
it does not overwrite the policy selected above. Explain how costs influence a recommendation.


In [ ]:
sensitivity = []
for penalty in [1.0, 3.0, 6.0]:
    table = capacity_scenarios(validation.cnt, validation.prediction, EXCESS_COST, penalty)
    best = table.loc[table.hypothetical_cost.idxmin()]
    sensitivity.append({'shortage_cost': penalty, 'selected_buffer_pct': int(best.buffer_pct),
                        'validation_cost': best.hypothetical_cost})
display(pd.DataFrame(sensitivity).round(2))


## What you should be able to explain
- Why a larger buffer lowers shortages but increases excess slots.
- Why a hypothetical cost is not a measured saving.
- Why the validation period selects the buffer and test only evaluates it.
- Why rental counts cannot directly determine physical fleet inventory.

**Next:** `04_review.ipynb` inspects errors and reloads the saved model.
